# Condition 2 Pilot Analysis (2026-03-11)

First pilot of condition 2 (associative recognition): n=6, all condition 2.

### Analyses

**Study phase (orienting).** 2(flanker gender compatibility: compatible/incompatible) × 3(flanker emotion: angry/happy/neutral) repeated-measures ANOVAs on accuracy and RT. Identical to condition 1.

**Test phase (associative recognition).** 2(pair type: intact/rearranged) × 3(flanker emotion: angry/happy/neutral) repeated-measures ANOVAs on p("same") and RT. Follow-up comparisons use the omnibus error term.

**Supplementary.** Signal detection analysis (d', criterion) per emotion. Unlike condition 1, both hit and false alarm rates are computed per emotion since rearranged pairs retain their original emotion labels.

### Design recap

**Study phase.** Same as condition 1: neutral target flanked by emotional faces, participant judges target gender. 12 trial types × 5 reps = 60 trials.

**Test phase (condition 2 — associative recognition).** Each trial shows a face pair (target + flanker, both neutral). The pair is either intact (same target-flanker pairing as study) or rearranged (target paired with a different flanker of the same gender and emotion). The participant judges "same" or "different". Key mapping is counterbalanced.

**Note:** Due to a bug in the intact/rearranged allocation code, the actual split is 24 intact / 36 rearranged per subject (8/12 per emotion), rather than the intended 30/30.

In [1]:
import pandas as pd
from pathlib import Path
from statistics import NormalDist

z = NormalDist().inv_cdf

_nb_dir = Path.cwd() if '__vsc_ipynb_file__' not in dir() else Path(__vsc_ipynb_file__).parent
df = pd.read_csv(_nb_dir / '2026_03_11_pilot_6subj_c2.csv')
df['correct'] = df['correct'].astype('boolean')

print(f'{len(df)} rows, {df.subject_number.nunique()} subjects')
print(f'Conditions: {df.groupby("condition").subject_number.nunique().to_dict()}')
print()

# Verify test-phase split
test_all = df[df.phase == 'test']
pt_counts = test_all.groupby(['subject_number', 'pair_type']).size().unstack(fill_value=0)
print('Test-phase intact/rearranged per subject:')
print(pt_counts.to_string())
print()
emo_pt = test_all.groupby(['flanker_emotion', 'pair_type']).size().unstack(fill_value=0)
print('Test-phase counts by emotion x pair_type (all subjects):')
print(emo_pt.to_string())

720 rows, 6 subjects
Conditions: {2: 6}

Test-phase intact/rearranged per subject:
pair_type       intact  rearranged
subject_number                    
1                   24          36
2                   24          36
3                   24          36
4                   24          36
5                   24          36
6                   24          36

Test-phase counts by emotion x pair_type (all subjects):
pair_type        intact  rearranged
flanker_emotion                    
angry                48          72
happy                48          72
neutral              48          72


## Exclusion Criteria

Same criterion as condition 1: exclude subjects with ≥6 zero-correct cells out of 12 in the study-phase design (2 target gender × 2 flanker gender × 3 flanker emotion, 5 trials each).

In [2]:
ZERO_CELL_THRESHOLD = 6

study_all = df[df.phase == 'study']

cell_correct = study_all.groupby(
    ['subject_number', 'target_gender', 'flanker_gender', 'flanker_emotion']
).correct.sum().reset_index()
zero_cells = cell_correct.groupby('subject_number').apply(
    lambda g: (g.correct == 0).sum()
).reset_index(name='zero_cells')

subj_study_acc = study_all.groupby('subject_number').correct.mean()
zero_cells['study_accuracy'] = zero_cells.subject_number.map(subj_study_acc)

print(f'Study-phase design: 12 cells per subject (5 trials each)')
print(f'Subjects with zero-correct cells:')
has_zeros = zero_cells[zero_cells.zero_cells > 0].sort_values('zero_cells', ascending=False)
if len(has_zeros) == 0:
    print('  None')
else:
    for _, row in has_zeros.iterrows():
        flag = ' ** EXCLUDED' if row.zero_cells >= ZERO_CELL_THRESHOLD else ''
        print(f'  Subject {int(row.subject_number)}: '
              f'{int(row.zero_cells)}/12 zero cells, {row.study_accuracy:.1%} accuracy{flag}')
print()

excluded = zero_cells[zero_cells.zero_cells >= ZERO_CELL_THRESHOLD].subject_number.tolist()
keep = zero_cells[zero_cells.zero_cells < ZERO_CELL_THRESHOLD].subject_number.tolist()
df = df[df.subject_number.isin(keep)].copy()
print(f'{len(excluded)} excluded, {df.subject_number.nunique()} subjects remain')

Study-phase design: 12 cells per subject (5 trials each)
Subjects with zero-correct cells:
  Subject 5: 1/12 zero cells, 75.0% accuracy

0 excluded, 6 subjects remain


### Exclusion summary

No subjects excluded. One subject (5) has 1 zero-correct cell out of 12 with 75% overall study accuracy, well below the threshold of 6. All 6 subjects are retained for analysis.

## Study Phase (Orienting)

Identical to condition 1: 2(flanker gender compatibility) × 3(flanker emotion) RM ANOVAs on accuracy and RT.

In [3]:
study = df[df.phase == 'study'].copy()
study['compatible'] = study.target_gender == study.flanker_gender
study['compat_label'] = study.compatible.map({True: 'compatible', False: 'incompatible'})

study_acc_subj = study.groupby(
    ['subject_number', 'compat_label', 'flanker_emotion']
).correct.mean().reset_index(name='accuracy')

study_rt_subj = study[~study.timed_out].groupby(
    ['subject_number', 'compat_label', 'flanker_emotion']
).rt.mean().reset_index(name='mean_rt')

acc_table = study_acc_subj.groupby(['compat_label', 'flanker_emotion']).accuracy.agg(
    ['mean', 'std']
).round(3)
print('Study-phase accuracy by compatibility x emotion:')
print(acc_table.to_string())
print()

print('Marginal means (accuracy):')
print(f'  Compatible:   {study_acc_subj[study_acc_subj.compat_label == "compatible"].accuracy.mean():.3f}')
print(f'  Incompatible: {study_acc_subj[study_acc_subj.compat_label == "incompatible"].accuracy.mean():.3f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {study_acc_subj[study_acc_subj.flanker_emotion == emo].accuracy.mean():.3f}')
print()

rt_table = study_rt_subj.groupby(['compat_label', 'flanker_emotion']).mean_rt.agg(
    ['mean', 'std']
).round(1)
print('Study-phase RT (ms) by compatibility x emotion:')
print(rt_table.to_string())
print()

print('Marginal means (RT):')
print(f'  Compatible:   {study_rt_subj[study_rt_subj.compat_label == "compatible"].mean_rt.mean():.1f}')
print(f'  Incompatible: {study_rt_subj[study_rt_subj.compat_label == "incompatible"].mean_rt.mean():.1f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {study_rt_subj[study_rt_subj.flanker_emotion == emo].mean_rt.mean():.1f}')

Study-phase accuracy by compatibility x emotion:
                               mean    std
compat_label flanker_emotion              
compatible   angry            0.967  0.082
             happy             0.95  0.084
             neutral          0.983  0.041
incompatible angry            0.833  0.273
             happy            0.883  0.194
             neutral            0.9  0.089

Marginal means (accuracy):
  Compatible:   0.967
  Incompatible: 0.872
         angry: 0.900
         happy: 0.917
       neutral: 0.942

Study-phase RT (ms) by compatibility x emotion:
                               mean    std
compat_label flanker_emotion              
compatible   angry            928.4  250.6
             happy            922.2  237.2
             neutral          833.1  186.8
incompatible angry            972.6  303.9
             happy            975.6  240.6
             neutral          966.1  323.3

Marginal means (RT):
  Compatible:   894.6
  Incompatible: 971.4
         a

In [4]:
import math

# --- Exact p-values via incomplete beta function (no scipy needed) ---

def _betacf(a, b, x):
    MAXIT, EPS = 200, 3e-12
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c = 1.0
    d = 1.0 / (1.0 - qab * x / qap) if abs(1.0 - qab * x / qap) > 1e-30 else 1.0 / 1e-30
    h = d
    for m in range(1, MAXIT + 1):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30: d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30: c = 1e-30
        d = 1.0 / d
        h *= d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        if abs(d) < 1e-30: d = 1e-30
        c = 1.0 + aa / c
        if abs(c) < 1e-30: c = 1e-30
        d = 1.0 / d
        delta = d * c
        h *= delta
        if abs(delta - 1.0) < EPS:
            break
    return h

def _betai(a, b, x):
    if x <= 0: return 0.0
    if x >= 1: return 1.0
    lbeta = math.lgamma(a) + math.lgamma(b) - math.lgamma(a + b)
    front = math.exp(a * math.log(x) + b * math.log(1 - x) - lbeta)
    if x < (a + 1) / (a + b + 2):
        return front * _betacf(a, b, x) / a
    else:
        return 1.0 - front * _betacf(b, a, 1 - x) / b

def t_p_twotail(t_val, df):
    x = df / (df + t_val ** 2)
    return _betai(df / 2.0, 0.5, x)

def f_p(f_val, df1, df2):
    if f_val <= 0: return 1.0
    x = df2 / (df2 + df1 * f_val)
    return _betai(df2 / 2.0, df1 / 2.0, x)


def rm_anova_oneway(groups):
    k = len(groups)
    n = len(groups[0])
    grand = sum(sum(g) for g in groups) / (k * n)
    subj_m = [sum(groups[j][i] for j in range(k)) / k for i in range(n)]
    cond_m = [sum(g) / n for g in groups]
    ss_cond = n * sum((m - grand) ** 2 for m in cond_m)
    ss_subj = k * sum((m - grand) ** 2 for m in subj_m)
    ss_total = sum((groups[j][i] - grand) ** 2 for j in range(k) for i in range(n))
    ss_err = ss_total - ss_cond - ss_subj
    df1 = k - 1
    df2 = (k - 1) * (n - 1)
    ms_err = ss_err / df2 if df2 > 0 else float('nan')
    f_val = (ss_cond / df1) / ms_err if ss_err > 0 else float('nan')
    p = f_p(f_val, df1, df2)
    eta = ss_cond / (ss_cond + ss_err)
    return f_val, df1, df2, p, eta, ms_err


def rm_anova_twoway(data, a_levels, b_levels):
    a = len(a_levels)
    b = len(b_levels)
    n = len(data[(a_levels[0], b_levels[0])])
    Y = [[[data[(a_levels[j], b_levels[k])][i]
           for k in range(b)] for j in range(a)] for i in range(n)]
    gm = sum(Y[i][j][k] for i in range(n) for j in range(a) for k in range(b)) / (n * a * b)
    subj_m = [sum(Y[i][j][k] for j in range(a) for k in range(b)) / (a * b) for i in range(n)]
    a_m = [sum(Y[i][j][k] for i in range(n) for k in range(b)) / (n * b) for j in range(a)]
    b_m = [sum(Y[i][j][k] for i in range(n) for j in range(a)) / (n * a) for k in range(b)]
    ab_m = [[sum(Y[i][j][k] for i in range(n)) / n for k in range(b)] for j in range(a)]
    sa_m = [[sum(Y[i][j][k] for k in range(b)) / b for j in range(a)] for i in range(n)]
    sb_m = [[sum(Y[i][j][k] for j in range(a)) / a for k in range(b)] for i in range(n)]
    ss_a = n * b * sum((a_m[j] - gm) ** 2 for j in range(a))
    ss_b = n * a * sum((b_m[k] - gm) ** 2 for k in range(b))
    ss_ab = n * sum((ab_m[j][k] - a_m[j] - b_m[k] + gm) ** 2
                    for j in range(a) for k in range(b))
    ss_s = a * b * sum((subj_m[i] - gm) ** 2 for i in range(n))
    ss_as = b * sum((sa_m[i][j] - a_m[j] - subj_m[i] + gm) ** 2
                    for i in range(n) for j in range(a))
    ss_bs = a * sum((sb_m[i][k] - b_m[k] - subj_m[i] + gm) ** 2
                    for i in range(n) for k in range(b))
    ss_total = sum((Y[i][j][k] - gm) ** 2
                   for i in range(n) for j in range(a) for k in range(b))
    ss_abs = ss_total - ss_a - ss_b - ss_ab - ss_s - ss_as - ss_bs
    df_a, df_b, df_ab = a - 1, b - 1, (a - 1) * (b - 1)
    df_as, df_bs, df_abs = df_a * (n - 1), df_b * (n - 1), df_ab * (n - 1)
    results = {}
    for label, ss_eff, df_eff, ss_e, df_e in [
        ('A', ss_a, df_a, ss_as, df_as),
        ('B', ss_b, df_b, ss_bs, df_bs),
        ('AxB', ss_ab, df_ab, ss_abs, df_abs),
    ]:
        ms_eff = ss_eff / df_eff if df_eff > 0 else 0
        ms_e = ss_e / df_e if df_e > 0 else float('nan')
        f_val = ms_eff / ms_e if ms_e > 0 else float('nan')
        p = f_p(f_val, df_eff, df_e)
        eta = ss_eff / (ss_eff + ss_e) if (ss_eff + ss_e) > 0 else 0
        results[label] = {
            'F': f_val, 'df1': df_eff, 'df2': df_e,
            'p': p, 'eta_sq': eta, 'ms_error': ms_e
        }
    return results


def anova_followup(means_a, means_b, ms_error, df_error, label_a, label_b):
    n = len(means_a)
    diff = sum(a - b for a, b in zip(means_a, means_b)) / n
    se = math.sqrt(2 * ms_error / n)
    if se == 0:
        return 0.0, df_error, 1.0, diff
    t_val = diff / se
    p = t_p_twotail(t_val, df_error)
    return t_val, df_error, p, diff


def edge_correct(rate, n):
    if rate == 0:
        return 0.5 / n
    if rate == 1:
        return 1 - 0.5 / n
    return rate


# --- Study-phase 2x3 ANOVAs ---

subjects = sorted(study_acc_subj.subject_number.unique())
n_subj = len(subjects)
a_levels = ['compatible', 'incompatible']
b_levels = ['angry', 'happy', 'neutral']

acc_data = {}
for al in a_levels:
    for bl in b_levels:
        mask = (study_acc_subj.compat_label == al) & (study_acc_subj.flanker_emotion == bl)
        vals = study_acc_subj[mask].set_index('subject_number').loc[subjects, 'accuracy'].tolist()
        acc_data[(al, bl)] = vals

print(f'2(compatibility) x 3(emotion) RM ANOVA on accuracy (n={n_subj}):')
print()
acc_results = rm_anova_twoway(acc_data, a_levels, b_levels)
for label, name in [('A', 'Compatibility'), ('B', 'Emotion'), ('AxB', 'Compatibility x Emotion')]:
    r = acc_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

rt_data = {}
for al in a_levels:
    for bl in b_levels:
        mask = (study_rt_subj.compat_label == al) & (study_rt_subj.flanker_emotion == bl)
        vals = study_rt_subj[mask].set_index('subject_number').loc[subjects, 'mean_rt'].tolist()
        rt_data[(al, bl)] = vals

print(f'2(compatibility) x 3(emotion) RM ANOVA on RT (n={n_subj}):')
print()
rt_results = rm_anova_twoway(rt_data, a_levels, b_levels)
for label, name in [('A', 'Compatibility'), ('B', 'Emotion'), ('AxB', 'Compatibility x Emotion')]:
    r = rt_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")

2(compatibility) x 3(emotion) RM ANOVA on accuracy (n=6):

  Compatibility: F(1,5) = 1.841, p = 0.233, partial eta^2 = 0.269
  Emotion: F(2,10) = 0.514, p = 0.613, partial eta^2 = 0.093
  Compatibility x Emotion: F(2,10) = 0.302, p = 0.746, partial eta^2 = 0.057

2(compatibility) x 3(emotion) RM ANOVA on RT (n=6):

  Compatibility: F(1,5) = 5.108, p = 0.073, partial eta^2 = 0.505
  Emotion: F(2,10) = 2.264, p = 0.155, partial eta^2 = 0.312
  Compatibility x Emotion: F(2,10) = 0.867, p = 0.450, partial eta^2 = 0.148


### Study phase interpretation

With only 6 subjects, no effects reach significance. The patterns are directionally consistent with the condition 1 pooled analysis:

**Compatibility.** Compatible trials yield higher accuracy (.967 vs .872) and faster RT (895 vs 971 ms). The RT effect approaches significance: F(1,5) = 5.11, p = .073, partial eta-sq = .505. The accuracy effect is not significant (p = .233) but the effect size is moderate (eta-sq = .269). The lack of significance likely reflects low power (n = 6).

**Emotion and interaction.** Neither the emotion main effect nor the compatibility x emotion interaction approaches significance in either DV (all p > .15). In the condition 1 pooled sample (n = 32), the interaction was strong in accuracy (p = .001). This sample is far too small to detect it.

## Test Phase (Associative Recognition)

Each test trial shows a face pair. Intact pairs are the same target-flanker combination seen at study; rearranged pairs swap the flanker identity within the same trial type (preserving target gender, flanker gender, and flanker emotion). The participant judges "same" (intact) or "different" (rearranged).

We analyze p("same") and RT in a 2(pair type: intact/rearranged) × 3(flanker emotion: angry/happy/neutral) design. p("same") for intact trials is the hit rate; p("same") for rearranged trials is the false alarm rate. Discrimination is indicated by higher p("same") for intact than rearranged.

**Note on cell sizes.** Due to the 24/36 split bug, each subject contributes 8 intact and 12 rearranged trials per emotion.

In [5]:
test = df[df.phase == 'test'].copy()

# p("same") = proportion responding "same" (said intact)
# intact + correct → said "same" (hit)
# rearranged + not correct → said "same" (false alarm)
test['said_same'] = test.apply(
    lambda r: bool(r.correct) if r.pair_type == 'intact' else not bool(r.correct), axis=1
)

# Per-subject p("same") in each cell of 2x3 design
psame_subj = test.groupby(
    ['subject_number', 'pair_type', 'flanker_emotion']
).said_same.mean().reset_index(name='p_same')

# Per-subject RT in each cell
rt_test_subj = test[~test.timed_out].groupby(
    ['subject_number', 'pair_type', 'flanker_emotion']
).rt.mean().reset_index(name='mean_rt')

# Group means: p("same")
print('Test-phase p("same") by pair_type x emotion:')
psame_table = psame_subj.groupby(['pair_type', 'flanker_emotion']).p_same.agg(
    ['mean', 'std']
).round(3)
print(psame_table.to_string())
print()

print('Marginal means p("same"):')
print(f'  Intact:     {psame_subj[psame_subj.pair_type == "intact"].p_same.mean():.3f}')
print(f'  Rearranged: {psame_subj[psame_subj.pair_type == "rearranged"].p_same.mean():.3f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {psame_subj[psame_subj.flanker_emotion == emo].p_same.mean():.3f}')
print()

# Group means: RT
print('Test-phase RT (ms) by pair_type x emotion:')
rt_table = rt_test_subj.groupby(['pair_type', 'flanker_emotion']).mean_rt.agg(
    ['mean', 'std']
).round(1)
print(rt_table.to_string())
print()

print('Marginal means RT (ms):')
print(f'  Intact:     {rt_test_subj[rt_test_subj.pair_type == "intact"].mean_rt.mean():.1f}')
print(f'  Rearranged: {rt_test_subj[rt_test_subj.pair_type == "rearranged"].mean_rt.mean():.1f}')
for emo in ['angry', 'happy', 'neutral']:
    print(f'  {emo:>12}: {rt_test_subj[rt_test_subj.flanker_emotion == emo].mean_rt.mean():.1f}')

Test-phase p("same") by pair_type x emotion:
                             mean    std
pair_type  flanker_emotion              
intact     angry            0.417  0.102
           happy            0.458  0.188
           neutral          0.396  0.184
rearranged angry            0.431  0.162
           happy            0.486  0.144
           neutral          0.431  0.178

Marginal means p("same"):
  Intact:     0.424
  Rearranged: 0.449
         angry: 0.424
         happy: 0.472
       neutral: 0.413

Test-phase RT (ms) by pair_type x emotion:
                              mean    std
pair_type  flanker_emotion               
intact     angry            1076.2  405.5
           happy            1032.8  387.5
           neutral          1151.0  496.7
rearranged angry            1066.6  494.9
           happy            1075.5  461.6
           neutral          1042.9  425.5

Marginal means RT (ms):
  Intact:     1086.7
  Rearranged: 1061.7
         angry: 1071.4
         happy: 1054.1
 

In [6]:
# --- 2(pair_type) x 3(emotion) RM ANOVAs ---

test_subjects = sorted(psame_subj.subject_number.unique())
n_test = len(test_subjects)
pt_levels = ['intact', 'rearranged']
emo_levels = ['angry', 'happy', 'neutral']

# p("same")
psame_data = {}
for pt in pt_levels:
    for emo in emo_levels:
        mask = (psame_subj.pair_type == pt) & (psame_subj.flanker_emotion == emo)
        vals = psame_subj[mask].set_index('subject_number').loc[test_subjects, 'p_same'].tolist()
        psame_data[(pt, emo)] = vals

print(f'2(pair_type) x 3(emotion) RM ANOVA on p("same") (n={n_test}):')
print()
psame_results = rm_anova_twoway(psame_data, pt_levels, emo_levels)
for label, name in [('A', 'Pair type'), ('B', 'Emotion'), ('AxB', 'Pair type x Emotion')]:
    r = psame_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

# RT
rt_data_test = {}
for pt in pt_levels:
    for emo in emo_levels:
        mask = (rt_test_subj.pair_type == pt) & (rt_test_subj.flanker_emotion == emo)
        vals = rt_test_subj[mask].set_index('subject_number').loc[test_subjects, 'mean_rt'].tolist()
        rt_data_test[(pt, emo)] = vals

print(f'2(pair_type) x 3(emotion) RM ANOVA on RT (n={n_test}):')
print()
rt_test_results = rm_anova_twoway(rt_data_test, pt_levels, emo_levels)
for label, name in [('A', 'Pair type'), ('B', 'Emotion'), ('AxB', 'Pair type x Emotion')]:
    r = rt_test_results[label]
    print(f"  {name}: F({r['df1']},{r['df2']}) = {r['F']:.3f}, "
          f"p = {r['p']:.3f}, partial eta^2 = {r['eta_sq']:.3f}")
print()

# --- Follow-up comparisons ---

# Use the pair_type main effect MSE for intact vs rearranged comparisons
# Use the interaction MSE for within-level comparisons
mse_pt_psame = psame_results['A']['ms_error']
df_pt_psame = psame_results['A']['df2']
mse_emo_psame = psame_results['B']['ms_error']
df_emo_psame = psame_results['B']['df2']
mse_int_psame = psame_results['AxB']['ms_error']
df_int_psame = psame_results['AxB']['df2']

mse_pt_rt = rt_test_results['A']['ms_error']
df_pt_rt = rt_test_results['A']['df2']
mse_int_rt = rt_test_results['AxB']['ms_error']
df_int_rt = rt_test_results['AxB']['df2']

print('Follow-up comparisons on p("same"):')
print()
print('  Intact vs rearranged within each emotion (discrimination):')
for emo in emo_levels:
    t, dfe, p, md = anova_followup(
        psame_data[('intact', emo)], psame_data[('rearranged', emo)],
        mse_int_psame, df_int_psame, f'intact-{emo}', f'rearranged-{emo}'
    )
    print(f'    {emo}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('  Pairwise emotion within intact (hit rate modulation):')
emo_pairs = [('angry', 'happy'), ('angry', 'neutral'), ('happy', 'neutral')]
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        psame_data[('intact', e1)], psame_data[('intact', e2)],
        mse_int_psame, df_int_psame, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('  Pairwise emotion within rearranged (FA rate modulation):')
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        psame_data[('rearranged', e1)], psame_data[('rearranged', e2)],
        mse_int_psame, df_int_psame, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

print('Follow-up comparisons on RT:')
print()
print('  Intact vs rearranged within each emotion:')
for emo in emo_levels:
    t, dfe, p, md = anova_followup(
        rt_data_test[('intact', emo)], rt_data_test[('rearranged', emo)],
        mse_int_rt, df_int_rt, f'intact-{emo}', f'rearranged-{emo}'
    )
    print(f'    {emo}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')
print()

print('  Pairwise emotion within intact:')
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        rt_data_test[('intact', e1)], rt_data_test[('intact', e2)],
        mse_int_rt, df_int_rt, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')
print()

print('  Pairwise emotion within rearranged:')
for e1, e2 in emo_pairs:
    t, dfe, p, md = anova_followup(
        rt_data_test[('rearranged', e1)], rt_data_test[('rearranged', e2)],
        mse_int_rt, df_int_rt, e1, e2
    )
    print(f'    {e1} vs {e2}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.1f} ms')

2(pair_type) x 3(emotion) RM ANOVA on p("same") (n=6):

  Pair type: F(1,5) = 0.174, p = 0.694, partial eta^2 = 0.034
  Emotion: F(2,10) = 1.205, p = 0.340, partial eta^2 = 0.194
  Pair type x Emotion: F(2,10) = 0.042, p = 0.959, partial eta^2 = 0.008

2(pair_type) x 3(emotion) RM ANOVA on RT (n=6):

  Pair type: F(1,5) = 0.144, p = 0.720, partial eta^2 = 0.028
  Emotion: F(2,10) = 0.448, p = 0.651, partial eta^2 = 0.082
  Pair type x Emotion: F(2,10) = 1.593, p = 0.251, partial eta^2 = 0.242

Follow-up comparisons on p("same"):

  Intact vs rearranged within each emotion (discrimination):
    angry: t(10) = -0.268, p = 0.794, diff = -0.014
    happy: t(10) = -0.537, p = 0.603, diff = -0.028
    neutral: t(10) = -0.671, p = 0.517, diff = -0.035

  Pairwise emotion within intact (hit rate modulation):
    angry vs happy: t(10) = -0.805, p = 0.439, diff = -0.042
    angry vs neutral: t(10) = 0.403, p = 0.696, diff = 0.021
    happy vs neutral: t(10) = 1.208, p = 0.255, diff = 0.062

  Pa

### Test phase interpretation

**Subjects show no associative discrimination.** The pair type main effect on p("same") is completely null: F(1,5) = 0.17, p = .694. p("same") is actually slightly higher for rearranged pairs (.449) than intact pairs (.424), the opposite of what discrimination would predict. None of the within-emotion intact vs rearranged comparisons approach significance (all p > .50).

**No emotion effects.** Neither the emotion main effect (p = .340) nor the pair type x emotion interaction (p = .959) is significant on p("same"). The same holds for RT (all p > .25).

**RT patterns.** No systematic RT differences between pair types or emotions. One hint: intact-neutral trials are numerically slowest (1151 ms) while rearranged-neutral are fastest (1043 ms), yielding a 108 ms difference that approaches significance (p = .105). But this is likely noise given the small n and the null p("same") pattern.

**Interpretation.** Subjects cannot distinguish intact from rearranged pairs in this design. Several factors may contribute: (a) with only 5 study reps per trial type and the 3-second fixed display, memory for specific target-flanker pairings may be too weak to support associative recognition; (b) the rearrangement preserves trial type (target gender, flanker gender, flanker emotion), so intact and rearranged pairs are perceptually very similar at test; (c) the 24/36 imbalance (bug) means rearranged is more common, which could bias responding but should not eliminate discrimination if it exists; (d) n = 6 is small, but the effect sizes are near zero, suggesting this is not simply a power issue.

## Supplementary: Signal Detection Analysis

d' (sensitivity) and criterion c computed per subject per flanker emotion. Unlike condition 1, both hit and false alarm rates are available per emotion: hit = p("same" | intact), FA = p("same" | rearranged). Edge correction follows Macmillan & Kaplan (1985).

Cell sizes are small (8 intact, 12 rearranged per emotion per subject), making edge correction essential.

In [7]:
# SDT per emotion: hit = p(same|intact), FA = p(same|rearranged)
sdt_per_subj = []
for subj in test_subjects:
    sdata = test[test.subject_number == subj]
    for emotion in emo_levels:
        intact_emo = sdata[(sdata.pair_type == 'intact') & (sdata.flanker_emotion == emotion)]
        rearr_emo = sdata[(sdata.pair_type == 'rearranged') & (sdata.flanker_emotion == emotion)]

        n_intact = len(intact_emo)
        n_rearr = len(rearr_emo)

        hit_rate_raw = intact_emo.said_same.mean() if n_intact > 0 else 0.0
        fa_rate_raw = rearr_emo.said_same.mean() if n_rearr > 0 else 0.0

        hit_rate = edge_correct(hit_rate_raw, n_intact) if n_intact > 0 else 0.5
        fa_rate = edge_correct(fa_rate_raw, n_rearr) if n_rearr > 0 else 0.5

        dprime = z(hit_rate) - z(fa_rate)
        criterion = -0.5 * (z(hit_rate) + z(fa_rate))

        sdt_per_subj.append({
            'subject': subj,
            'emotion': emotion,
            'n_intact': n_intact,
            'n_rearranged': n_rearr,
            'hit_rate': hit_rate_raw,
            'fa_rate': fa_rate_raw,
            'd_prime': dprime,
            'criterion': criterion
        })

sdt_df = pd.DataFrame(sdt_per_subj)

sdt_summary = sdt_df.groupby('emotion').agg(
    N=('subject', 'count'),
    hit_rate_M=('hit_rate', 'mean'),
    hit_rate_SD=('hit_rate', 'std'),
    fa_rate_M=('fa_rate', 'mean'),
    fa_rate_SD=('fa_rate', 'std'),
    d_prime_M=('d_prime', 'mean'),
    d_prime_SD=('d_prime', 'std'),
    criterion_M=('criterion', 'mean'),
    criterion_SD=('criterion', 'std'),
).round(3)

print(f'SDT Analysis (n={n_test} subjects, per-emotion hit and FA rates)')
print(f'Cell sizes: {sdt_df.n_intact.iloc[0]} intact, {sdt_df.n_rearranged.iloc[0]} rearranged per emotion per subject')
print()
print(sdt_summary.to_string())
print()

# One-way RM ANOVA on d'
dprime_wide = sdt_df.pivot(index='subject', columns='emotion', values='d_prime')
angry_d = dprime_wide['angry'].tolist()
happy_d = dprime_wide['happy'].tolist()
neutral_d = dprime_wide['neutral'].tolist()

f_val_d, df1_d, df2_d, p_d, eta_d, mse_d = rm_anova_oneway([angry_d, happy_d, neutral_d])
print(f"One-way RM ANOVA on d' (flanker emotion):")
print(f"  F({df1_d},{df2_d}) = {f_val_d:.3f}, p = {p_d:.3f}, partial eta^2 = {eta_d:.3f}")
print()

# Follow-ups
print("Follow-up comparisons on d' (using omnibus MSE):")
d_groups = [angry_d, happy_d, neutral_d]
d_labels = ['angry', 'happy', 'neutral']
for i in range(3):
    for j in range(i + 1, 3):
        t, dfe, p, md = anova_followup(d_groups[i], d_groups[j], mse_d, df2_d,
                                        d_labels[i], d_labels[j])
        print(f'  {d_labels[i]} vs {d_labels[j]}: t({dfe}) = {t:.3f}, p = {p:.3f}, diff = {md:.3f}')
print()

# Also one-way ANOVA on criterion
crit_wide = sdt_df.pivot(index='subject', columns='emotion', values='criterion')
angry_c = crit_wide['angry'].tolist()
happy_c = crit_wide['happy'].tolist()
neutral_c = crit_wide['neutral'].tolist()

f_val_c, df1_c, df2_c, p_c, eta_c, mse_c = rm_anova_oneway([angry_c, happy_c, neutral_c])
print(f"One-way RM ANOVA on criterion (flanker emotion):")
print(f"  F({df1_c},{df2_c}) = {f_val_c:.3f}, p = {p_c:.3f}, partial eta^2 = {eta_c:.3f}")

SDT Analysis (n=6 subjects, per-emotion hit and FA rates)
Cell sizes: 8 intact, 12 rearranged per emotion per subject

         N  hit_rate_M  hit_rate_SD  fa_rate_M  fa_rate_SD  d_prime_M  d_prime_SD  criterion_M  criterion_SD
emotion                                                                                                     
angry    6       0.417        0.102      0.431       0.162     -0.031       0.481        0.203         0.268
happy    6       0.458        0.188      0.486       0.144     -0.072       0.589        0.077         0.338
neutral  6       0.396        0.184      0.431       0.178     -0.108       0.330        0.250         0.486

One-way RM ANOVA on d' (flanker emotion):
  F(2,10) = 0.077, p = 0.927, partial eta^2 = 0.015

Follow-up comparisons on d' (using omnibus MSE):
  angry vs happy: t(10) = 0.209, p = 0.838, diff = 0.041
  angry vs neutral: t(10) = 0.392, p = 0.703, diff = 0.077
  happy vs neutral: t(10) = 0.183, p = 0.859, diff = 0.036

One-way RM ANOV

### SDT interpretation

d' is near zero or slightly negative for all emotions (angry = -0.03, happy = -0.07, neutral = -0.11), confirming the complete absence of associative discrimination. These values are indistinguishable from chance. The d' ANOVA is null: F(2,10) = 0.08, p = .927.

The criterion values are slightly positive (angry = 0.20, happy = 0.08, neutral = 0.25), indicating a mild bias toward responding "different" (saying the pair is rearranged). This could partly reflect the 24/36 base rate (60% of test pairs are rearranged), though it could also reflect uncertainty defaulting to "different."

The condition 1 d' values (0.40-0.47) were already low. The condition 2 d' values near zero suggest that associative recognition is substantially harder than item recognition in this paradigm.

## Summary

### Practical implications

**Associative recognition appears too difficult in the current design.** Condition 1 item recognition d' was already weak (~0.40-0.47). Condition 2 associative recognition d' is at zero. Subjects cannot distinguish intact from rearranged pairs.

**The 24/36 split bug must be fixed regardless.** The gender-code mismatch in `trials.js` produces 24 intact / 36 rearranged instead of 30/30. This biases the response criterion (subjects may default to "different" when uncertain, matching the higher base rate of rearranged trials), but it should not eliminate discrimination if subjects have any memory for the pairings. Still, it should be corrected before collecting more data.

**Why the task may be too hard.** Rearrangement within trial type preserves target gender, flanker gender, and flanker emotion. The only difference between an intact and rearranged pair is the specific identity of the flanker. With 5 study reps per trial type, subjects may not encode which particular flanker went with which particular target. Item recognition (condition 1) only requires recognizing a face as previously seen; associative recognition requires remembering the specific pairing, which is a harder memory demand.

**Possible design changes to discuss with Gordon:**
- More study repetitions per pair, to strengthen associative memory traces
- More distinctive rearrangements (e.g., swapping across emotions or genders rather than within trial type), at the cost of the within-trial-type control
- A different encoding task that promotes relational processing of the target-flanker pair
- Whether to continue collecting condition 2 data or pause pending design revision

**Caveats.** n = 6 is very small. However, the effect sizes are near zero (pair type eta-sq = .034, d' values slightly negative), which is informative: even accounting for low power, there is no hint of discrimination in the expected direction. A larger sample could reveal very weak discrimination, but the current data do not encourage optimism.

In [8]:
n_total = 6
n_excluded = n_total - df.subject_number.nunique()
n_kept = df.subject_number.nunique()

print(f'=== Condition 2 Pilot Summary (2026-03-11) ===')
print(f'{n_total} subjects, {n_excluded} excluded (>=6 zero cells), {n_kept} analyzed')
print(f'Test split: 24 intact / 36 rearranged per subject (bug: intended 30/30)')
print()

print('Study phase (orienting):')
print(f'  Overall accuracy: {study.correct.mean():.1%}')
print(f'  Mean RT: {study.loc[~study.timed_out, "rt"].mean():.0f} ms')
r = acc_results['A']
print(f"  Compatibility: F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
r = acc_results['B']
print(f"  Emotion:       F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
r = acc_results['AxB']
print(f"  Interaction:   F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}")
print()

print('Test phase (associative recognition):')
r = psame_results['A']
print(f"  p(\"same\") Pair type:       F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = psame_results['B']
print(f"  p(\"same\") Emotion:          F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = psame_results['AxB']
print(f"  p(\"same\") Interaction:      F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = rt_test_results['A']
print(f"  RT Pair type:              F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = rt_test_results['B']
print(f"  RT Emotion:                F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
r = rt_test_results['AxB']
print(f"  RT Interaction:            F({r['df1']},{r['df2']}) = {r['F']:.3f}, p = {r['p']:.3f}, eta^2 = {r['eta_sq']:.3f}")
print()

print('Supplementary SDT:')
for emotion, row in sdt_summary.iterrows():
    print(f"  {emotion:>7}: d'={row['d_prime_M']:.2f} (SD={row['d_prime_SD']:.2f}), "
          f"c={row['criterion_M']:.2f}, hit={row['hit_rate_M']:.2f}, fa={row['fa_rate_M']:.2f}")
print(f"  d' ANOVA: F({df1_d},{df2_d}) = {f_val_d:.3f}, p = {p_d:.3f}, eta^2 = {eta_d:.3f}")

=== Condition 2 Pilot Summary (2026-03-11) ===
6 subjects, 0 excluded (>=6 zero cells), 6 analyzed
Test split: 24 intact / 36 rearranged per subject (bug: intended 30/30)

Study phase (orienting):
  Overall accuracy: 91.9%
  Mean RT: 932 ms
  Compatibility: F(1,5) = 1.841, p = 0.233
  Emotion:       F(2,10) = 0.514, p = 0.613
  Interaction:   F(2,10) = 0.302, p = 0.746

Test phase (associative recognition):
  p("same") Pair type:       F(1,5) = 0.174, p = 0.694, eta^2 = 0.034
  p("same") Emotion:          F(2,10) = 1.205, p = 0.340, eta^2 = 0.194
  p("same") Interaction:      F(2,10) = 0.042, p = 0.959, eta^2 = 0.008
  RT Pair type:              F(1,5) = 0.144, p = 0.720, eta^2 = 0.028
  RT Emotion:                F(2,10) = 0.448, p = 0.651, eta^2 = 0.082
  RT Interaction:            F(2,10) = 1.593, p = 0.251, eta^2 = 0.242

Supplementary SDT:
    angry: d'=-0.03 (SD=0.48), c=0.20, hit=0.42, fa=0.43
    happy: d'=-0.07 (SD=0.59), c=0.08, hit=0.46, fa=0.49
  neutral: d'=-0.11 (SD=0.33)